In [ ]:
!nvidia-smi

In [ ]:
import os, sys, subprocess, torch

In [ ]:
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

!pip install torch-geometric --quiet
!pip install pyg-lib torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-{torch.__version__}.html \
    --only-binary :all: \
    --quiet \
    || echo "Legacy extensions skipped."

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
!pip install torch_geometric --quiet

In [ ]:
# CELL 4 — Dataset builder with corrected oracle (margin=45, not 110)
import json, math, os, torch
from torch_geometric.data import Data
from torch.utils.data import Dataset
from tqdm.auto import tqdm

MANEUVER_NAMES = ["NORMAL", "BARREL_ROLL", "JINKING", "FALLING_LEAF", "COBRA", "IMMELMANN"]
MANEUVER_IDX   = {n: i for i, n in enumerate(MANEUVER_NAMES)}

class PrebuiltDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self):         return len(self.data)
    def __getitem__(self, i):  return self.data[i]

# ── CORRECTED ORACLE (margin 110→45) ─────────────────────────────────────────
# The sim bounces the jet at 30px. margin=110 labeled 68% of frames IMMELMANN.
# margin=45 = bounce_zone(30) + 15px buffer — correct danger zone.
def oracle_maneuver_corrected(jet_pos, missiles, w=800, h=600, margin=45):
    if not missiles: return 0   # NORMAL
    jx, jy = jet_pos
    if jx < margin or jx > w-margin or jy < margin or jy > h-margin:
        return 5   # IMMELMANN

    near      = [(m, math.hypot(m["pos"][0]-jx, m["pos"][1]-jy)) for m in missiles]
    close_80  = [m for m,d in near if d <  80]
    close_150 = [m for m,d in near if d < 150]
    close_200 = [m for m,d in near if d < 200]

    if len(close_80) == 1:
        m  = close_80[0]
        dx = m["pos"][0]-jx; dy = m["pos"][1]-jy
        if m["vel"][0]*dx + m["vel"][1]*dy < 0: return 4  # COBRA

    if len(close_150) >= 2: return 3  # FALLING_LEAF

    if len(missiles) >= 3 and len(close_200) >= 1: return 2  # JINKING

    if (len(close_200) == 1 and len(missiles) == 1
            and min(jx, w-jx) > 160 and min(jy, h-jy) > 160):
        return 1  # BARREL_ROLL

    return 0  # NORMAL


def build_record(record):
    jet      = record["jet"]
    missiles = record["missiles"]
    flares   = record["flares"]
    label    = torch.tensor([record["label"]], dtype=torch.float)

    # ── Always re-compute maneuver with corrected oracle ──────────────────────
    # Ignores the wrong stored "maneuver" value from old jsonl files
    maneuver_label = torch.tensor(
        oracle_maneuver_corrected(jet["pos"], missiles), dtype=torch.long
    )

    sub_graphs = []
    for m in missiles:
        nearby_f = [f for f in flares
                    if math.hypot(f["pos"][0]-m["pos"][0],
                                  f["pos"][1]-m["pos"][1]) < 200][:6]
        nodes = (
            [{"pos": jet["pos"], "vel": jet["vel"], "type": 0},
             {"pos": m["pos"],   "vel": m["vel"],   "type": 1}]
            + [{"pos": f["pos"], "vel": f["vel"], "type": 2} for f in nearby_f]
        )
        x_rows, pos_t, vel_t = [], [], []
        for node in nodes:
            p, v = node["pos"], node["vel"]
            x_rows.append(p + v + [float(node["type"])])
            pos_t.append(torch.tensor(p, dtype=torch.float))
            vel_t.append(torch.tensor(v, dtype=torch.float))

        x  = torch.tensor(x_rows, dtype=torch.float)
        n  = len(nodes)
        ei, ea = [], []
        for i in range(n):
            for j in range(n):
                if i == j: continue
                ei.append([i, j])
                ea.append(torch.cat([pos_t[i]-pos_t[j], vel_t[i]-vel_t[j]]))
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.stack(ea).float()

        jx, jy   = jet["pos"]; jvx, jvy = jet["vel"]
        mx, my   = m["pos"];   mvx, mvy = m["vel"]
        dx, dy   = jx-mx, jy-my
        dist     = math.hypot(dx, dy) + 1e-5
        nd       = min(dist/800.0, 1.0)
        rvx, rvy = mvx-jvx, mvy-jvy
        closing  = (dx*rvx + dy*rvy)/dist
        cn       = max(-1.0, min(1.0, closing/10.0))
        tti      = min((dist/closing)/60.0, 1.0) if closing > 0.05 else 1.0
        mhl      = math.hypot(rvx, rvy) + 1e-5
        aoa      = max(0.0, (dx*rvx + dy*rvy)/(dist*mhl))
        mtype    = float(m.get("type", 0))
        ctx      = torch.tensor([nd, tti, cn, aoa, mtype], dtype=torch.float)

        sub_graphs.append(Data(x=x, edge_index=edge_index,
                               edge_attr=edge_attr, context=ctx))

    if not sub_graphs: return None
    return (sub_graphs,
            torch.stack([g.context for g in sub_graphs]),
            label,
            maneuver_label)


In [ ]:
def collate_fn(batch):
    sub_graphs_list = [item[0] for item in batch]
    contexts_list   = [item[1] for item in batch]
    labels          = torch.stack([item[2] for item in batch])
    maneuver_labels = torch.stack([item[3] for item in batch])
    return sub_graphs_list, contexts_list, labels, maneuver_labels


In [ ]:
import torch
import torch.nn as nn
from torch_geometric.nn import MetaLayer
from torch_geometric.data import Data, Batch
from torch_geometric.utils import scatter

class EdgeModel(nn.Module):
    def __init__(self, node_dim=64, edge_dim=32, msg_dim=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(node_dim*2 + edge_dim, 128), nn.ReLU(), nn.LayerNorm(128),
            nn.Linear(128, 64), nn.ReLU(), nn.LayerNorm(64),
            nn.Linear(64, msg_dim), nn.ReLU(), nn.LayerNorm(msg_dim)
        )
    def forward(self, src, dest, edge_attr, u, batch):
        return self.mlp(torch.cat([src, dest, edge_attr], dim=-1))

class NodeModel(nn.Module):
    def __init__(self, node_dim=64, msg_dim=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(node_dim + msg_dim, 128), nn.ReLU(), nn.LayerNorm(128),
            nn.Linear(128, 64), nn.ReLU(), nn.LayerNorm(64),
            nn.Linear(64, node_dim), nn.ReLU(), nn.LayerNorm(node_dim)
        )
    def forward(self, x, edge_index, edge_attr, u, batch):
        row, col = edge_index
        agg = scatter(edge_attr, col, dim=0, dim_size=x.size(0), reduce="sum")
        return self.mlp(torch.cat([x, agg], dim=-1))

class ThreatGNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.node_enc  = nn.Sequential(nn.Linear(5, 64), nn.ReLU(), nn.LayerNorm(64))
        self.edge_enc  = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.LayerNorm(32))
        self.layers    = nn.ModuleList([
            MetaLayer(EdgeModel(64, 32, 32), NodeModel(64, 32)),
            MetaLayer(EdgeModel(64, 32, 32), NodeModel(64, 32)),
            MetaLayer(EdgeModel(64, 32, 32), NodeModel(64, 32)),
        ])
        self.pool_head = nn.Sequential(nn.Linear(64, 32), nn.ReLU())

    def forward(self, data):
        x, ei, ea, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x  = self.node_enc(x)
        ea = self.edge_enc(ea)
        for layer in self.layers:
            x, ea, _ = layer(x, ei, ea, None, batch)
        pooled = scatter(x, batch, dim=0, reduce="mean")
        return self.pool_head(pooled)   # [K, 32]

class TemporalThreatAttention(nn.Module):
    def __init__(self, embed_dim=32, context_dim=5, hidden_dim=64):
        super().__init__()
        self.hidden_dim  = hidden_dim
        self.gru         = nn.GRUCell(embed_dim + context_dim, hidden_dim)
        self.attn_scorer = nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.ReLU(), nn.LayerNorm(32), nn.Linear(32, 1)
        )
        self.danger_head = nn.Sequential(
            nn.Linear(hidden_dim, 16), nn.ReLU(), nn.LayerNorm(16),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, embeds, contexts, hidden=None):
        if embeds.size(0) == 0:
            dev = embeds.device
            return torch.tensor([0.0], device=dev), torch.empty(0, device=dev), None
        K = embeds.size(0)
        if hidden is None or hidden.size(0) != K:
            hidden = torch.zeros(K, self.hidden_dim, device=embeds.device)
        combined     = torch.cat([embeds, contexts], dim=-1)     # [K, 37]
        new_hidden   = self.gru(combined, hidden)                  # [K, 64]
        scores       = self.attn_scorer(new_hidden)                # [K, 1]
        attn_weights = torch.softmax(scores / 0.5, dim=0)
        z            = torch.sum(attn_weights * new_hidden, dim=0, keepdim=True)  # [1, 64]
        danger       = self.danger_head(z).squeeze(-1)             # [1]
        return danger, attn_weights.squeeze(-1), new_hidden

class NeuralEvasionBrain(nn.Module):
    def __init__(self):
        super().__init__()
        self.threat_gnn    = ThreatGNN()
        self.attention     = TemporalThreatAttention()
        # Input: top-attended GRU hidden [64] + danger scalar [1] = 65
        self.maneuver_head = nn.Sequential(
            nn.Linear(65, 32), nn.ReLU(), nn.LayerNorm(32),
            nn.Linear(32, 6)   # 6 maneuver classes
        )

    def forward(self, sub_graphs_list, contexts, hidden=None):
        if not sub_graphs_list:
            dev = contexts.device if contexts is not None else torch.device('cpu')
            return (torch.tensor([0.0], device=dev),
                    torch.empty(0, device=dev),
                    None,
                    torch.zeros(1, 6, device=dev))

        batched = Batch.from_data_list(sub_graphs_list)
        embeds  = self.threat_gnn(batched)                         # [K, 32]
        danger, attn_w, new_hidden = self.attention(embeds, contexts, hidden)

        # Maneuver: concat hidden of most-attended missile + danger scalar
        top_idx  = attn_w.argmax().clamp(0, new_hidden.size(0)-1)
        top_h    = new_hidden[top_idx]                             # [64]
        m_input  = torch.cat([top_h, danger.detach().reshape(1)], dim=0)  # [65]
        m_logits = self.maneuver_head(m_input.unsqueeze(0))        # [1, 6]

        return danger, attn_w, new_hidden, m_logits

print("Architecture loaded. Param count:")
_m = NeuralEvasionBrain()
print(f"  Total: {sum(p.numel() for p in _m.parameters()):,}")
del _m


In [ ]:
import time
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np

def forward_batched(model, batch_sub_graphs, batch_contexts, device):
    """
    Mega-batch ThreatGNN across ALL scenes in one GPU call.
    GRU still runs per-scene (stateful), but it's tiny (K=1..8 missiles).
    Result: ~5x faster than calling model() once per scene.
    """
    # Flatten all sub_graphs + record scene boundaries
    all_flat, scene_sizes = [], []
    for sgs in batch_sub_graphs:
        all_flat.extend([g.to(device) for g in sgs])
        scene_sizes.append(len(sgs))

    if not all_flat:
        return None, None

    # ONE ThreatGNN forward pass for every missile in the entire batch
    mega    = Batch.from_data_list(all_flat)
    embeds  = model.threat_gnn(mega)   # [total_missiles, 32]

    all_preds, all_m_logits = [], []
    cursor = 0
    for i, K in enumerate(scene_sizes):
        e   = embeds[cursor:cursor+K]                      # [K, 32]
        ctx = batch_contexts[i].to(device)                 # [K, 5]

        # GRU attention (hidden=None: independent per scene, no cross-batch leakage)
        danger, attn_w, hidden = model.attention(e, ctx, hidden=None)

        # Maneuver head
        top_idx  = attn_w.argmax().clamp(0, hidden.size(0)-1)
        top_h    = hidden[top_idx]
        m_input  = torch.cat([top_h, danger.detach().reshape(1)], dim=0)
        m_logits = model.maneuver_head(m_input.unsqueeze(0))   # [1, 6]

        all_preds.append(danger.reshape(1))
        all_m_logits.append(m_logits)
        cursor += K

    preds    = torch.cat(all_preds).unsqueeze(-1)      # [B, 1]
    m_logits = torch.cat(all_m_logits, dim=0)          # [B, 6]
    return preds, m_logits


In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader
import random, time, json, os
import torch.optim as optim
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

STAGE_FILES = [
    'data_slow_singles.jsonl',
    'data_fast_homing.jsonl',
    'data_multi_threat.jsonl',
    'data_cluster_swarm.jsonl',
    'data_adversarial.jsonl',
]

# Fallback to single file if curriculum not uploaded
if not os.path.exists(STAGE_FILES[0]):
    print('WARNING: Curriculum files not found. Falling back to data.jsonl')
    STAGE_FILES = ['data.jsonl']

model     = NeuralEvasionBrain().to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                  patience=2, factor=0.5)

danger_criterion   = nn.BCELoss(reduction='none')
maneuver_criterion = nn.CrossEntropyLoss()

stage_auc_history  = []
best_val_auc_global = 0.0

for stage_idx, stage_file in enumerate(STAGE_FILES):
    print(f'\n{"="*54}')
    print(f'STAGE {stage_idx+1}: {stage_file}')
    print(f'{"="*54}')

    # ── Stage cache: build once, reload instantly ──────────────
    cache_file = stage_file.replace('.jsonl', '_cache.pt')

    if os.path.exists(cache_file):
        print(f'  Loading cached graphs from {cache_file}...')
        stage_data = torch.load(cache_file)
        print(f'  Loaded {len(stage_data)} cached records')
    else:
        raw = []
        with open(stage_file) as f:
            for line in f:
                line = line.strip()
                if line:
                    try: raw.append(json.loads(line))
                    except: pass
        print(f'  Loaded {len(raw)} raw records')

        stage_data = []
        for record in tqdm(raw, desc='Building graphs'):
            result = build_record(record)
            if result: stage_data.append(result)
        print(f'  Built {len(stage_data)} graph records')

        torch.save(stage_data, cache_file)
        print(f'  Cached to {cache_file}')

    # Label + maneuver distribution
    n1 = sum(1 for _,_,lbl,_ in stage_data if lbl.item() == 1)
    n0 = len(stage_data) - n1
    pos_weight = torch.tensor(n0/max(n1,1), dtype=torch.float).to(device)
    print(f'  Label 0: {n0} | Label 1: {n1} | pos_weight: {pos_weight.item():.2f}x')

    m_counts = [0]*len(MANEUVER_NAMES)
    for _,_,_,ml in stage_data: m_counts[ml.item()] += 1
    print('  Maneuver labels: ' + ' | '.join(
        f'{MANEUVER_NAMES[i]}={m_counts[i]}' for i in range(len(MANEUVER_NAMES))))

    # Warn if IMMELMANN is still dominant (shouldn't be with fixed oracle)
    imm_pct = 100*m_counts[5]/max(len(stage_data),1)
    if imm_pct > 30:
        print(f'  WARNING: IMMELMANN is {imm_pct:.1f}% of labels. Oracle may still be wrong.')

    # ── Split & loaders ────────────────────────────────────────
    random.shuffle(stage_data)
    split    = int(0.85 * len(stage_data))
    train_ds = PrebuiltDataset(stage_data[:split])
    val_ds   = PrebuiltDataset(stage_data[split:])

    # num_workers=0: safest on Colab with custom collate + GRU
    # batch_size=256: doubles throughput vs 128
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True,
                              collate_fn=collate_fn, num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False,
                              collate_fn=collate_fn, num_workers=0, pin_memory=False)

    best_auc     = 0.0
    patience     = 0
    MAX_PATIENCE = 3
    MAX_EPOCHS   = 8

    for epoch in range(1, MAX_EPOCHS+1):
        t0 = time.time()

        # ── TRAIN ──────────────────────────────────────────────
        model.train()
        train_losses = []
        for batch in train_loader:
            sg_list, ctx_list, labels, m_labels = batch
            labels   = labels.to(device)
            m_labels = m_labels.to(device)

            optimizer.zero_grad()
            preds, m_logits = forward_batched(model, sg_list, ctx_list, device)
            if preds is None: continue

            lbl_sq = labels.squeeze(-1)
            d_loss = (danger_criterion(preds.squeeze(-1), lbl_sq)
                      * pos_weight ** lbl_sq).mean()
            m_loss     = maneuver_criterion(m_logits, m_labels)
            total_loss = d_loss + 0.4 * m_loss

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(total_loss.item())

        # ── VAL ────────────────────────────────────────────────
        model.eval()
        val_preds_all, val_labels_all = [], []
        with torch.no_grad():
            for batch in val_loader:
                sg_list, ctx_list, labels, _ = batch
                labels = labels.to(device)
                preds, _ = forward_batched(model, sg_list, ctx_list, device)
                if preds is None: continue
                val_preds_all.extend(preds.squeeze(-1).cpu().numpy())
                val_labels_all.extend(labels.squeeze(-1).cpu().numpy())

        val_preds_arr  = np.array(val_preds_all)
        val_labels_arr = np.array(val_labels_all)
        try:
            val_auc = roc_auc_score(val_labels_arr, val_preds_arr)
        except Exception:
            val_auc = 0.0
        val_f1 = f1_score(val_labels_arr, (val_preds_arr >= 0.5).astype(int),
                          zero_division=0)

        avg_loss = np.mean(train_losses) if train_losses else 0.0
        scheduler.step(avg_loss)
        elapsed = int(time.time() - t0)

        print(f'  Epoch {epoch:02d} | Loss: {avg_loss:.4f} | '
              f'Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f} | {elapsed}s')

        stage_auc_history.append((stage_idx+1, epoch, val_auc))

        if val_auc > best_auc:
            best_auc     = val_auc
            patience     = 0
            if val_auc > best_val_auc_global:
                best_val_auc_global = val_auc
                torch.save(model.state_dict(), 'jet_brain_v3.pt')
                print(f'    ✓ New global best → jet_brain_v3.pt  (AUC: {val_auc:.4f})')
        else:
            patience += 1
            if patience >= MAX_PATIENCE:
                print(f'  Plateaued at stage {stage_idx+1}. Advancing.')
                break

print(f'\nTraining complete. Best AUC: {best_val_auc_global:.4f}')
print('Weights saved to jet_brain_v3.pt')


In [ ]:
import matplotlib.pyplot as plt

aucs   = [a for _,_,a in stage_auc_history]
stages = [s for s,_,_ in stage_auc_history]

plt.figure(figsize=(14, 4))
colors = ["#e74c3c","#e67e22","#f1c40f","#2ecc71","#9b59b6"]
plt.plot(aucs, marker="o", ms=4, linewidth=1.5, color="steelblue")

prev_i, prev_s = 0, stages[0] if stages else 1
for i, s in enumerate(stages):
    if s != prev_s or i == len(stages)-1:
        plt.axvspan(prev_i, i, alpha=0.08, color=colors[(prev_s-1)%5],
                    label=f"Stage {prev_s}")
        prev_i, prev_s = i, s
    if i > 0 and stages[i] != stages[i-1]:
        plt.axvline(x=i, color="salmon", linewidth=0.9, linestyle="--", alpha=0.7)
        plt.text(i+0.2, min(aucs)+0.005 if aucs else 0, f"S{stages[i]}", fontsize=8, color="salmon")

plt.title("Curriculum Learning — Val AUC per Epoch", fontsize=13)
plt.xlabel("Epoch (across all stages)")
plt.ylabel("Validation AUC")
if aucs:
    plt.ylim(max(0, min(aucs)-0.02), min(1.0, max(aucs)+0.02))
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig("curriculum_curve.png", dpi=150)
plt.show()
print(f"\nFinal best val AUC: {best_val_auc_global:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np, random

def visualize_attention(model, dataset, num_samples=3):
    model.eval()
    multi = [(i, *dataset[i][:3]) for i in range(len(dataset))
             if len(dataset[i][0]) > 1][:60]
    if not multi:
        print("No multi-missile scenes found."); return

    samples = random.sample(multi, min(num_samples, len(multi)))
    fig, axes = plt.subplots(len(samples), 2, figsize=(14, 6*len(samples)))
    if len(samples) == 1: axes = [axes]
    fig.patch.set_facecolor("#0a0e17")

    with torch.no_grad():
        for row, (idx, sub_graphs, contexts, _) in enumerate(samples):
            sg_dev  = [g.to(device) for g in sub_graphs]
            ctx_dev = contexts.to(device)
            # Correct 4-value API
            danger, attn_w, _, m_logits = model(sg_dev, ctx_dev, hidden=None)
            attn_np      = attn_w.cpu().numpy()
            danger_score = danger.item()
            rec_manu     = MANEUVER_NAMES[m_logits.argmax().item()]

            jet_pos      = [sub_graphs[0].x[0][0].item(), sub_graphs[0].x[0][1].item()]
            missiles_pos = [[g.x[1][0].item(), g.x[1][1].item()] for g in sub_graphs]

            ax_r, ax_b = axes[row]
            ax_r.set_facecolor("#121826")
            ax_r.scatter(*jet_pos, c="#00FFC8", s=250, marker="^",
                         edgecolors="white", linewidth=1, label="AI Jet")
            ax_r.text(jet_pos[0]+15, jet_pos[1], "AI Jet",
                      color="#00FFC8", fontsize=11, weight="bold")
            for mi, (mx, my) in enumerate(missiles_pos):
                w = attn_np[mi] if mi < len(attn_np) else 0
                ax_r.scatter(mx, my, c="#FF4444",
                             s=80 + w*300,          # size proportional to attention
                             marker="o", edgecolors="white", linewidth=0.5)
                ax_r.plot([jet_pos[0], mx], [jet_pos[1], my],
                          color="#FF4444", linestyle="--",
                          alpha=0.2 + w*0.8,         # opacity proportional to attention
                          linewidth=0.8 + w*2)
                ax_r.text(mx+10, my+10, f"M{mi+1}\n{w:.2f}",
                          color="white", fontsize=9, weight="bold")
            ax_r.set_xlim(0, 800); ax_r.set_ylim(0, 600); ax_r.invert_yaxis()
            ax_r.set_title(f"Scene {idx} | Danger: {danger_score:.3f} | GNN→{rec_manu}",
                           color="#38bdf8", fontsize=12)
            ax_r.tick_params(colors="#94a3b8")
            ax_r.grid(True, linestyle=":", alpha=0.5, color="#334155")

            ax_b.set_facecolor("#121826")
            dists = contexts[:, 0].cpu().numpy()
            bar_colors = ["#10b981" if i == int(np.argmin(dists)) else "#334155"
                          for i in range(len(attn_np))]
            ax_b.bar([f"M{i+1}" for i in range(len(attn_np))], attn_np,
                     color=bar_colors, edgecolor="white", linewidth=0.5)
            ax_b.set_ylim(0, 1.0)
            ax_b.set_title("GNN Attention Weights (green = closest)",
                           color="#38bdf8", fontsize=12)
            ax_b.tick_params(colors="#94a3b8")
            ax_b.grid(axis="y", linestyle=":", alpha=0.5, color="#334155")

    plt.tight_layout(); plt.savefig("attention_viz.png", dpi=120); plt.show()

val_dataset = PrebuiltDataset(stage_data[split:])
visualize_attention(model, val_dataset, num_samples=3)
